In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import spikeinterface.full as si
import spikeinterface.extractors as se
from spikeinterface.sortingcomponents.peak_detection import detect_peaks
from spikeinterface.sortingcomponents.peak_localization import localize_peaks
from pathlib import Path

# ---- !!! CHANGE THESE TWO FOR A NEW PATIENT !!! ----
name = "NPIX_MSIT_014"
folder = Path(r"C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014")
# ---------------------------------------------------

# output folder for this PREPROCESSING code
out_dir = folder / f"{name}_PREPROCESSING"
out_dir.mkdir(exist_ok=True)

# task computer's TTL log: search the session folder
# this will be used to align all plots/slices etc.. to the unified time base
ttl_logs = list(folder.rglob("*_TTL_log.txt"))
print(ttl_logs) # should be exactly one file
ttl_log_path = ttl_logs[0]

# load the recording
raw_AP = se.read_openephys(folder, stream_name="Record Node 123#OneBox-122.ProbeA-AP")
raw_LFP = se.read_openephys(folder, stream_name="Record Node 123#OneBox-122.ProbeA-LFP")
print(raw_AP)
print(raw_LFP)

# shared clock: time of the first sample of continuous.dat
t0 = raw_AP.get_start_time()
print(t0, raw_LFP.get_start_time())

# check SI's times match timestamps.npy (should be close to 0, which means no lost samples)
ts = np.load(folder / "Record Node 123" / "experiment1" / "recording1" / "continuous" / "OneBox-122.ProbeA-AP" / "timestamps.npy")
print(np.max(np.abs(ts - raw_AP.get_times())))

%matplotlib inline

[WindowsPath('C:/Users/wongbe/Desktop/PROJECTS_PC/npix_data_bhw/2026-10-02_15-08-00_NPIX_MSIT_014/NPX_MSIT_014_taskdata_good/NPX_MSIT_014_second_2026-10-02_15H_23m_51s_TTL_log.txt')]
OpenEphysBinaryRecordingExtractor: 384 channels - 30.0kHz - 1 segments - 48,159,900 samples 
                                   1,605.33s (26.76 minutes) - int16 dtype - 34.45 GiB
OpenEphysBinaryRecordingExtractor: 384 channels - 2500.0Hz - 1 segments - 4,013,325 samples 
                                   1,605.33s (26.76 minutes) - int16 dtype - 2.87 GiB
72.2996 72.2996
1.4642864698544145e-10


First we bandpass filter, phase shift, and common median reference the AP, only to find bad channels. This is just for initial inspection. 

Previously, in DJD's NeuropixelsDataPipelinesOverview, motion is estimated by DREDge on the LFP (bandpass, derivative, average, resample). The cleaned AP (bandpass, phase shift, CMR, notch-comb) is then motion-corrected in SpikeInterface with interpolate_motion and sorted with KS4.

For Eric's pipeline (which now has phase shifting as of 10/2026), motion is estimated by DREDge on the LFP (bandpass, phase shift, resample, average, in dredge_recording.py). Then you derive motion (keep only slow drift, at whatever gain you choose), then make_segment cuts a segment of the raw AP continuous.dat. MATLAB phase shifts it and then applies the kriging interpolation. The output then goes to KS4, which does its own filtering (high pass) and CAR.

In [3]:

# now band pass filter 300-3000 Hz for AP
AP_high = si.bandpass_filter(raw_AP, freq_min=300, freq_max=3000, margin_ms=20)

In [4]:
# phase shift BEFORE Common MEDIAN Reference
AP_phasecorrect = si.phase_shift(recording=AP_high)  # phase correct the recording

AP_CMR = si.common_reference(recording=AP_phasecorrect)

print(AP_CMR)

CommonReferenceRecording: 384 channels - 30.0kHz - 1 segments - 48,159,900 samples 
                          1,605.33s (26.76 minutes) - int16 dtype - 34.45 GiB


Previously, we used a scouting DREDge run on the full recording in order to estimate when the probe was inserted/settled. It seems easier to just use information from the TTL (on the unified clock) to find the task start, trials start and end, and then just DREDge from a border around those. That's what we will do below.

In [5]:
# find task start, trial start and end, based on task TTLs (the shared clock)
# this is similar to what we did in raw_data_explore, just getting experiment info to inform what we put into DREDge.

events = se.read_openephys_event(folder).get_events("OneBox ADC Digital Lines") # spike interface calls it this for some reason, it's the folder events\OneBox-122.OneBox-ADC\TTL
task_pulses = events["time"][events["label"] == "5"] # line 5 = task TTLs (line 1 = 1 Hz sync)

task_log_df = pd.read_table(ttl_log_path, sep=r"\s+") # task computer list of events, there is one row per pulse it sends. pd.read_table makes a copy of the log

## basically, the task_log from the task computer has a list of events, but no times. The task_pulses are recorded by OEphys, with unified time, but do not have event names.
# so we have to match task_log and task_pulses
task_log_df["npix_time"] = task_pulses[-len(task_log_df):] # last (len) pulses = the (len) rows of the log





# task start = first row of the log (first pulse of the start train)
# Note that this is NOT raw_AP.get_start_time(): that is when the recording started, not the task
task_start = task_log_df.npix_time.iloc[0]
print(task_log_df.Event.iloc[0], task_start) # check the event name is the start train

trial_start = task_log_df.npix_time[task_log_df.Event == "Fixation_Shown"].iloc[0]
trial_end = task_log_df.npix_time[task_log_df.Event == "Trial_Data_Stored"].iloc[-1]
print(trial_start, trial_end)

TTL_Start_Train_01 771.537204473174
982.5070630898542 1614.223793494684


We will then also identify bad channels that we want to exclude.

In [6]:
# find bad channels to remove from candidates below
AP_task_hp = AP_phasecorrect.frame_slice(start_frame=AP_phasecorrect.time_to_sample_index(trial_start),
                                         end_frame=AP_phasecorrect.time_to_sample_index(trial_end))

bad_ids, labels = si.detect_bad_channels(AP_task_hp)
print(bad_ids)
print(np.unique(labels, return_counts=True))

['AP64' 'AP191']
(array(['dead', 'good'], dtype='<U5'), array([  2, 382]))


In [7]:
# DREDge window
# will need to convert this to the continuous.dat file time because that's what Eric's code uses
# we will slice on either side of the first and last trial, but want to be careful not to cut too far to where we were still inserting the probe, which has lots of motion artifact
pad_before = 20 # s, room before the first trial without reaching insertion, try to gauge this from raster in raw data explore?
pad_after = 2 # s, small for MSIT 014

dredge_start = trial_start - pad_before
dredge_end = trial_end + pad_after
print("shared clock:", round(dredge_start, 1), round(dredge_end, 1))

# Eric's dredge_recording.py wants FILE time (0 = first sample), so subtract t0
print("file time (for Eric's script):", round(dredge_start - t0, 1), round(dredge_end - t0, 1))


shared clock: 962.5 1616.2
file time (for Eric's script): 890.2 1543.9


Now that we have the DREDge window we can run the DREDge, which will be in a Powershell:
cd C:\Users\wongbe\Desktop\PROJECTS_PC\NpxNonrigidInterp
$env:NPX_RECORDINGS = "C:\Users\wongbe\Desktop\PROJECTS_PC\npix_motion_bhw\recordings_bhw.json"
$env:NPX_RESULTS    = "C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014\NPIX_MSIT_014_motion_data\results"
$env:NPX_REC        = "msit014"

^ for example for MSIT014, change as needed

**refer to motion_interp_procedure for what to do next**



Output for MSIT014: msit014 fmin=0.5: 192 ch, 0-3820 um, t=890-1544 s (163425 frames @ 250 Hz)

=== msit014 / nonrigid_800 === fmin=0.5 {'rigid': False, 'win_step_um': 800.0, 'win_scale_um': 850.0}
  315 s -> (163425, 4), 4 window(s) at [ 710 1510 2310 3110]
  strain: median 0.106, 0.0% > 0.5
  SD 137.1 um   RMS>0.1Hz 67.6 um   cardiac-band RMS 49.8 um
  saved msit014_nonrigid_800_t890-1543

DONE

Next ran: cd C:\Users\wongbe\Desktop\PROJECTS_PC\npix_motion_bhw
python plot_dredge_scouting.py "C:/Users/wongbe/Desktop/PROJECTS_PC/npix_data_bhw/2026-10-02_15-08-00_NPIX_MSIT_014/NPIX_MSIT_014_motion_data/msit014_nonrigid_800_t890-1543_displacement.csv" **--mark 910.2 --mark 1541.9**

^ remember this plot is still in continuous.dat file time, so for marks of trial start and trial end, use the above trial_start and trial_end times minus t0 (72.3 for MSIT014). 

![alt text](msit014_nonrigid_800_t890-1543_motion.png)

We see that there is some motion, as well as some sharp changes (downward spikes at ~1000, ~1250, ~1300, ~1400, etc)
This is what derive_motion fixes by filtering out large changes.

In [8]:
# ---- SEGMENT WINDOW for make_segment below, use to find correct continuous.dat times ----
# Recall:
# 1. Eric's scripts take CONTINUOUS.DAT FILE time (0 = first sample of continuous.dat).
#    file time = TTL/shared/unified clock time - t0      (recall t0 = raw_AP.get_start_time())
# 2. The segment must sit INSIDE the DREDge window, or make_segment no work
# 3. The segment should cover ALL trials (trial_start to trial_end), plus a small pad.
# 4. If rules 2 and 3 conflict, try to widen the DREDge window and re-run DREDge, or accept losing the edge trials.

seg_pad = 5 # seconds of extra data you want around the trials

# shared clock: trials with pad, but clip to stay inside the DREDge window
seg_start = max(trial_start - seg_pad, dredge_start) # take the later of the two start times, ie if the padding goes before DREDge start, just take the DREDge start
seg_end   = min(trial_end + seg_pad, dredge_end) # take the earlier of the two end times, if padding would run past DREDge, just use DREDge end.

# CONVERT TO CONTINUOUS.DAT FILE TIME - ceil the start and floor the end so rounding can't push it outside
seg_start_file = np.ceil(seg_start - t0)
seg_end_file   = np.floor(seg_end - t0)

print("segment, shared clock:", round(seg_start, 1), round(seg_end, 1))
print("segment, CONTINUOUS.DAT FILE time (for make_segment):", seg_start_file, seg_end_file)
print("covers all trials:", seg_start <= trial_start and seg_end >= trial_end)


segment, shared clock: 977.5 1616.2
segment, CONTINUOUS.DAT FILE time (for make_segment): 906.0 1543.0
covers all trials: True


So later on, we'll need to use this make_segment time to calculate where to look in Segment time, because where Segment time = 0, is whatever the make_segment (continuous.dat) start time is. 

If using the plot_traces widget from RAW_DATA_EXPLORE_BHW, do this: **desired time from plot_traces (shared time) - get_start_time (72.29 for MSIT014) - make_segment time (906 in this case)**

Next: cd C:\Users\wongbe\Desktop\PROJECTS_PC\NpxNonrigidInterp

 slow drift from 0 to 0.1, keep gain at 0.6 for this time just to see, and then rename the segment to t890 so we can keep it separate from the earlier one. 
python scripts\derive_motion.py msit014_nonrigid_800_t890-1543 **msit014_t890_g060** "0,0.1,0.6"

outputs this: msit014_nonrigid_800_t890-1543 -> msit014_t890_g060   bands [[0.0, 0.1, 0.6]]   median SD 137.1 -> 72.6 um

 then cut the raw AP segment (continuous.dat time) + its motion trace:
    to do this, 
python scripts\make_segment.py 906 1543 seg_t890_g060 msit014_t890_g060

^ the DREDge started at 890, the segment started at 906, I'm just using DREDge for the time to keep thigns less confusing. 



Then, python scripts\check_segment.py "C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014\Record Node 123\experiment1\recording1\continuous\OneBox-122.ProbeA-AP\seg_t890_g060"


Outputs: 
segment C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014\Record Node 123\experiment1\recording1\continuous\OneBox-122.ProbeA-AP\seg_t890_g060
  interp_params.json  {'motion_fs': 250, 'batch_samples': 128, 'nchan_raw': 384, 'm_tol_um': 0.5, 'phase_shift': True, 'phase_shift_margin_ms': 40.0, 'out_name': 'interpseg_t890_g060'}
  displacement        4 window(s) x 159254 samples = 637.0 s at 250 Hz, starting at t = 906.0 s
                      range -387.0 to 60.5 um, per-window SD [51.1 68.7 90.8 72.9] um
  window centres      [ 710. 1510. 2310. 3110.] um
  reference           [ -72.1 -135.6 -196.5 -150.2] um
  max strain          0.176 between neighbouring windows
  ADC phase shift     ON, 40 ms margin, 384 channels, skew 28.2 us max
  continuous.dat      14676480000 bytes = 19110000 frames x 384 ch = 637.0 s
  chanMap.mat         384 sites, y 0-3820 um (pitch [20.]), x [27. 59.]
OK - ready for matlab/run_interp.m


Next, put through matlab, can do via same terminal or in matlab itself, here is via terminal:
& "C:\Program Files\MATLAB\R2026b\bin\matlab.exe" -batch "addpath('C:\Users\wongbe\Desktop\PROJECTS_PC\NpxNonrigidInterp\matlab'); run_interp('C:\Users\wongbe\Desktop\PROJECTS_PC\npix_data_bhw\2026-10-02_15-08-00_NPIX_MSIT_014\Record Node 123\experiment1\recording1\continuous\OneBox-122.ProbeA-AP\seg_t890_g060', 'C:\Users\wongbe\Desktop\PROJECTS_PC\InterpolationAfterDREDge')"


Then ran KS4 on the interp file

- Make sure there is a kilosort4 folder within the interpseg folder

In a new terminal: deactivate
& C:\Users\wongbe\Desktop\PROJECTS_PC\bhw_code\phy-env.venv\Scripts\Activate.ps1
